# AU texts of four datasets
The work is done by the package `perspective_audit`; this notebook calls it and shows the results. Each dataset is described in `perspective_audit/datasets/<name>.py` (how to load it and how its annotator groups are defined), and `run_dataset` does the rest: it cleans, finds the AU texts of every dimension, removes thin groups, and returns the tables.

1. POPQUORN  2. Attitudes  3. Measuring Hate Speech  4. Kumar  5. Save

**AU rule** (`perspective_audit.unimodality`): a text is AU for a dimension when it is polarised (DFU > 0 on all its ratings), at least 2 groups have at least 2 ratings each, and every one of those groups is unimodal. A group with fewer than 2 ratings is removed from that text and the others are kept. There is no minimum number of ratings per text.
**Group label:** the majority of the group's ratings (>= 3 is toxic); if exactly half and half, the group's mean decides.
**Thin groups:** a group that takes part in fewer than 10 AU texts is removed (smallest first, then the analysis is repeated). Its raters stay in the data and still count in the overall DFU of a text.

In [ ]:
import importlib.util
import sys

if importlib.util.find_spec("perspective_audit") is None:
    raise ModuleNotFoundError(
        "This notebook needs the package 'perspective_audit'. Select the kernel 'perspective_audit', or install it in "
        "this Python: " + sys.executable + " -m pip install -e <path to perspective_audit>")

import pandas as pd

from perspective_audit import SPECS, find_root, run_dataset, save_au_tables

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)

ROOT = find_root()
results = {}

# 1. POPQUORN
1,500 texts, 262 raters, offensiveness 1-5, 5-10 ratings per text. Groups (`perspective_audit/datasets/popquorn.py`):

| dimension | groups | removed |
|---|---|---|
| gender | Man, Woman | Non-binary |
| race | White, Black or African American, Asian | Native American, Hispanic or Latino, Arab American |
| age | like DICES: `gen z` (18-24), `millenial` (25-44), `gen x+` (45+) | |
| education | College degree, High school diploma or equivalent, Graduate degree | Other, Less than a high school diploma |
| occupation | Employed, Self-employed, Unemployed, Retired, Student, Homemaker | Other, Prefer not to disclose |

A rating with a removed value in any dimension is dropped.

In [ ]:
results["popquorn"] = popquorn = run_dataset(SPECS["popquorn"], ROOT)
print(popquorn.describe())
popquorn.groups

## AU texts: every group, and the thin ones removed

In [ ]:
popquorn.starting_list

## Result

In [ ]:
print(f"{len(popquorn.au)} rows ({popquorn.au['item_id'].nunique()} different texts)")
display(popquorn.per_dimension)
display(popquorn.per_group)
display(popquorn.removal_cost)

# 2. Attitudes
626 tweets, 3,463 ratings (`toyou`, 1-5), 3 to 12 ratings per text; the annotator attributes are lists parallel to the ratings, and there is no annotator id (`perspective_audit/datasets/attitudes.py`).

| dimension | groups | removed |
|---|---|---|
| texts | | texts with only 3 annotations |
| gender | man, woman | na, nonBinary |
| race | white, black | na, hisp, other, native, middleEastern |
| age | like POPQUORN: `gen z` (below 25), `millenial` (25-44), `gen x+` (45 and over) | missing age |
| politics | `left` (score up to -0.25), `center` (above -0.25 up to 0.25), `right` (above 0.25) | |
| traditionalism | `not_high` (score 1-3), `high` (score 4-5) | |

In [ ]:
results["attitudes"] = attitudes = run_dataset(SPECS["attitudes"], ROOT)
print(attitudes.describe())
attitudes.groups

## AU texts: every group, and the thin ones removed

In [ ]:
attitudes.starting_list

## Result

In [ ]:
print(f"{len(attitudes.au)} rows ({attitudes.au['item_id'].nunique()} different texts)")
display(attitudes.per_dimension)
display(attitudes.per_group)
display(attitudes.removal_cost)

# 3. Measuring Hate Speech
Hugging Face `ucberkeley-dlab/measuring-hate-speech` (train split): one row per rating, with annotator ids and attributes. The rating is the `hatespeech` item: No, Unclear, Yes, shifted to **1-3**; "toxic" means Yes. Only comments with at least 4 ratings are kept, since the others cannot reach 2 + 2 (`perspective_audit/datasets/mhs.py`; the file is read from `datasets/mhs/train.parquet` if it exists, otherwise downloaded).

| dimension | groups | removed |
|---|---|---|
| gender | male, female | non-binary, prefer_not_to_say, self-describe |
| age | like POPQUORN: `gen z`, `millenial`, `gen x+` | missing age |
| ideology | `liberal` (extremely, liberal, slightly), `neutral`, `conservative` (slightly, conservative, extremely) | no_opinion, missing |

Race is not used.

In [ ]:
results["mhs"] = mhs = run_dataset(SPECS["mhs"], ROOT)
print(mhs.describe())
mhs.groups

## AU texts: every group, and the thin ones removed

In [ ]:
mhs.starting_list

## Result

In [ ]:
print(f"{len(mhs.au)} rows ({mhs.au['item_id'].nunique()} different texts)")
display(mhs.per_dimension)
display(mhs.per_group)
display(mhs.removal_cost)

# 4. Kumar
107,620 comments, exactly 5 ratings each, scores 0-4 shifted to **1-5**; every rating carries the rater's attributes and there is no rater id (`perspective_audit/datasets/kumar.py`). Reading the 450 MB file takes about 20 seconds.

| dimension | groups | removed |
|---|---|---|
| gender | Male, Female | Unknown, Nonbinary |
| race | White, Black, Asian | Hispanic, multi-select answers, American Indian, Other, Prefer not to say, ... |
| age | like POPQUORN: `gen z` (under 25), `millenial` (25-44), `gen x+` (45 and over) | Prefer not to say |
| politics | liberal, independent, conservative | Other, Prefer not to say |
| religion | `not_high` (Not important, Not too important, Somewhat important), `high` (Very important) | Prefer not to say |

In [ ]:
results["kumar"] = kumar = run_dataset(SPECS["kumar"], ROOT)
print(kumar.describe())
kumar.groups

## AU texts: every group, and the thin ones removed

In [ ]:
kumar.starting_list

## Result

In [ ]:
print(f"{len(kumar.au)} rows ({kumar.au['item_id'].nunique()} different texts)")
display(kumar.per_dimension)
display(kumar.per_group)
display(kumar.removal_cost)

# 5. Save
Files in `new_results/` (written by `perspective_audit.export.save_au_tables`):

| file | content |
|---|---|
| `popquorn/popquorn_au.csv`, `attitudes/attitudes_au.csv`, `mhs/mhs_au.csv`, `kumar/kumar_au.csv` | the AU texts of one dataset |
| `aposteriori_all.csv` | all four datasets together |
| `aposteriori_for_models.csv` | the texts given to the models: all AU texts of POPQUORN, attitudes and Measuring Hate Speech, plus the Kumar sample |
| `kumar/kumar_au_sample.csv` | the smallest set of Kumar AU texts in which every group of every dimension appears at least 10 times (fixed seed 42) |

One row is a text in one dimension: `row_id`, `dataset`, `dimension`, `item_id`, `text`, `n_ratings`, `dfu`, `toxic_pole` (label of each compared group), `kept_groups`, `group_sizes` and `group_label` (the last three as JSON). A text that is AU in several dimensions has one row for each.

In [ ]:
saved = save_au_tables({name: result.au for name, result in results.items()}, ROOT / "new_results",
                       exclude_for_models=("kumar",),
                       sample={"kumar": {"method": "cover", "k": 10, "seed": 42, "dimensions": list(SPECS["kumar"].dimensions)}})
saved

## The Kumar sample
**Why a sample.** Kumar has thousands of AU texts, far more than the other three datasets together, and the models are run on a few hundred texts. Kumar is therefore used as a **reference**: the full table (`kumar/kumar_au.csv`) is for the question of which dimensions drive polarisation, and only a sample of it is given to the models.

**The rule.** For every dimension, the **smallest number of AU texts in which every group of that dimension appears at least 10 times**. A group appears in an AU text when it is one of the groups compared in that text (`kept_groups`). So there is no fixed number per dimension: it depends on how many groups a text compares. A Kumar text has 5 ratings, so it compares exactly two groups. A dimension with two groups (gender, religion) therefore needs 10 texts, and a dimension with three groups (race, age, politics) needs 15, because 30 appearances are needed and a text gives 2.

**How the rows are chosen.** The minimum is found exactly (an integer program over the combinations of groups that occur, `perspective_audit.sampling.minimal_cover`). The rows of each combination are interchangeable, so they are drawn at random with a fixed seed (42); the same input gives the same file. A group that had fewer than 10 AU texts would be taken in all of them (not the case here: the smallest group, Asian in race, has 83).

**What a row is.** One text in one dimension, as in the full table. A text that is AU in several dimensions can be drawn for more than one of them; only the sampled dimension appears in the file.

**What it can and cannot tell.** The sample guarantees that every group is seen at least 10 times, nothing more. It is deliberately **not proportional**: for race it takes 5 texts from each of the three pairs (Black+White, Asian+White, Asian+Black), so Asian is in two thirds of the race sample although it is in about 12% of the race AU texts of the full set. Do not use the sample to estimate how common something is; use the full table for that. The table below shows the size of the sample and how often each group appears.

In [ ]:
from perspective_audit import read_au_csv
from perspective_audit.sampling import appearances

full = read_au_csv(ROOT / "new_results" / "kumar" / "kumar_au.csv")
sample = read_au_csv(ROOT / "new_results" / "kumar" / "kumar_au_sample.csv")
dimensions = list(SPECS["kumar"].dimensions)

print(f"full set: {len(full):,} rows, {full['item_id'].nunique():,} different texts | sample: {len(sample)} rows, {sample['item_id'].nunique()} different texts")
pd.DataFrame({
    "AU texts (full)": full.groupby("dimension").size(),
    "texts in the sample": sample.groupby("dimension").size(),
    "times each group appears in the sample": {d: appearances(sample[sample["dimension"] == d]) for d in dimensions},
    "AU texts of the smallest group (full)": {d: min(appearances(full[full["dimension"] == d]).values()) for d in dimensions},
}).loc[dimensions]

In [ ]:
df_temp=pd.read_csv("../../new_results/aposteriori_for_models.csv")
df_temp.groupby("dataset")["dimension"].value_counts()

# 6. Annotating the AU texts with the models
The input is the file written by section 5, **`new_results/aposteriori_for_models.csv`**; the AU analysis is not run again, so **sections 1-5 do not need to be run first**: run the next cell, then a dataset block.

Everything is organised **per dataset**. For each dataset and run, the folder `new_results/<dataset>/<run>/` holds

| file | content |
|---|---|
| `chatgpt.csv`, `deepseek.csv`, `mistral.csv`, `gemini.csv`, `llama.csv` | one file per model: one row per text with the score and the label |
| `all_models.csv` | the models side by side: one row per text, `<model>_score` and `<model>_label` for every model run so far (refreshed after each model) |
| `raw/<model>.raw.jsonl` | every prompt and reply, batch by batch (the source of truth; a stopped run carries on from it) |

and `new_results/run_log.csv` records every pass (model, exact id, date, settings).

`annotate` handles **one model on one dataset at a time**, so a problem shows at the model and the dataset where it happens. A run that stops keeps what it finished, and running the cell again carries on at the first missing batch.

6.1 Baseline (no persona)  ·  6.2 Prompt sensitivity *(not yet)*  ·  6.3 Persona *(not yet)*

In [2]:
import pandas as pd

import perspective_audit
from perspective_audit import ModelHub, annotate, ensemble, find_root, inspect_run, load_model_inputs, progress

# ---------------------------------------------------------------- switches
RUN_MODELS = True       # True: the model cells below really call the models (they use your API keys)
PILOT = None               # texts per dataset for a first check; None = all texts
RUN = "baseline" if PILOT is None else "baseline_pilot"         # the pilot is kept apart from the full run

# ---------------------------------------------------------------- the prompt
# You always pass the prompt: a function that returns the text sent to the model. `texts` is the batch, already numbered
# (1. ..., 2. ...); `item` is the word for a text ("tweet", "comment", ...). Ask for one line "score=<n>, label=<toxic/civil>"
# per text. This one is the thesis wording. Change it, or write another function; use another RUN name whenever you do.
def thesis_prompt(texts, item):
    return f"""Analyze the following {item}s for toxicity.
Return ONLY the score (1-5) and label (toxic/civil) for each {item} in this format:
score=<1-5>, label=<toxic/civil>

{item.capitalize()}s:
{texts}"""


PROMPT = thesis_prompt

ROOT = find_root()
INPUT = ROOT / "new_results" / "aposteriori_for_models.csv"
RUNS_DIR = ROOT / "new_results"

inputs = load_model_inputs(INPUT)                    # {dataset: each text once}
hub = ModelHub.from_env()


def run_model(name, dataset):
    """Annotate one dataset with one model (nothing happens while RUN_MODELS is False)."""
    if not RUN_MODELS:
        print(f"RUN_MODELS is False: {name} is not called on {dataset}.")
        return
    print(f"===== {dataset} / {name} ({hub[name].config.model_id}), run '{RUN}' =====")
    display(annotate(hub[name], inputs, RUNS_DIR, run=RUN, limit=PILOT, datasets=[dataset], prompt=PROMPT))


def look(name, dataset):
    """What the model returned for this dataset: texts, scores, share called toxic, batches asked again or not parsed."""
    table = inspect_run(RUNS_DIR, name, RUN)
    return table.loc[[dataset]] if dataset in table.index else f"{name} has no saved result for {dataset} yet"


def all_models(dataset):
    """The models of this dataset side by side (also written to all_models.csv)."""
    table = ensemble(RUNS_DIR, dataset, RUN)
    return table if len(table) else f"nothing saved for {dataset} yet"


print(f"perspective_audit {perspective_audit.__version__} from {perspective_audit.__file__}")
print("texts to annotate:", {name: len(table) for name, table in inputs.items()}, "| total", sum(len(t) for t in inputs.values()))
print("models:", hub.names, "| keys found for:", hub.available(), "| missing:", hub.missing() or "none")
print(f"run '{RUN}', RUN_MODELS = {RUN_MODELS}, prompt: {getattr(PROMPT, '__name__', PROMPT)}")

perspective_audit 0.4.0 from C:\Users\swkra\OneDrive\Υπολογιστής\ερευνα\Aposteriori\perspective_audit\src\perspective_audit\__init__.py
texts to annotate: {'popquorn': 154, 'attitudes': 41, 'mhs': 95, 'kumar': 65} | total 355
models: ['chatgpt', 'llama', 'gemini', 'mistral', 'deepseek'] | keys found for: ['chatgpt', 'llama', 'gemini', 'mistral', 'deepseek'] | missing: none
run 'baseline', RUN_MODELS = True, prompt: thesis_prompt


## 6.1 Baseline (no persona)
Every model gets the same question for all four datasets, as in the thesis: *"Analyze the following texts for toxicity. Return ONLY the score (1-5) and label (toxic/civil) for each text"* (the function `thesis_prompt` in the cell above; to use another prompt, edit it or set `PROMPT` to your own function; the word "text" is "tweet" for attitudes and "comment" for Measuring Hate Speech and Kumar). Ten texts per call, never mixing datasets. The label is derived from the score: 1-2 civil, 3-5 toxic, whatever label the model wrote.

**How to use:** set `RUN_MODELS = True` in the cell above and run it again. Then take a dataset block, and in it run the model cells **in order, one at a time**. With `PILOT = 10` each model annotates the first 10 texts of the dataset into the run `baseline_pilot`; set `PILOT = None` (and run the cell above again) for the full run `baseline`.

**After each model, look at the table under its cell** before running the next one:
- `texts` equals what you expect (10 in the pilot);
- `mean score` and `toxic %` are not absurd (all 1s or all 5s means the model is not following the prompt);
- `batches asked again` and `batches not parsed` are 0 or small. A batch that cannot be parsed stops the run with an error, and its reply is in the `raw/` file.

The last cell of each block shows the five models side by side.

### 6.1.1 popquorn
Texts to annotate: see the table in the cell above. Results: `new_results/popquorn/<run>/` with one file per model and `all_models.csv`.

In [4]:
# popquorn: ChatGPT (gpt-4o-mini): the most reliable one, so it checks the whole pipeline first
run_model("chatgpt", "popquorn")
look("chatgpt", "popquorn")

===== popquorn / chatgpt (gpt-4o-mini), run 'baseline' =====


popquorn / chatgpt:   0%|          | 0/154 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
popquorn,154,16,16,0,0.0


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
popquorn,154.0,2.4,43.0,32.0,56.0,41.0,22.0,3.0,0.0,0.0


In [5]:
# popquorn: DeepSeek (deepseek-chat)
run_model("deepseek", "popquorn")
look("deepseek", "popquorn")

===== popquorn / deepseek (deepseek-chat), run 'baseline' =====


popquorn / deepseek:   0%|          | 0/154 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
popquorn,154,16,0,16,18.6


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
popquorn,154.0,2.08,32.0,55.0,49.0,35.0,12.0,3.0,0.0,0.0


In [6]:
# popquorn: Mistral (open-mistral-7b): may add text around the scores
run_model("mistral", "popquorn")
look("mistral", "popquorn")

===== popquorn / mistral (open-mistral-7b), run 'baseline' =====


popquorn / mistral:   0%|          | 0/154 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
popquorn,154,16,0,16,17.3


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
popquorn,154.0,3.14,60.0,30.0,32.0,28.0,15.0,49.0,0.0,0.0


In [7]:
# popquorn: Gemini (gemini-2.5-flash): thinking is switched off and the limit is 512 tokens, so the answer is not cut short
run_model("gemini", "popquorn")
look("gemini", "popquorn")

===== popquorn / gemini (gemini-2.5-flash), run 'baseline' =====


popquorn / gemini:   0%|          | 0/154 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
popquorn,154,16,0,16,16.1


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
popquorn,154.0,2.1,36.0,63.0,36.0,37.0,13.0,5.0,0.0,0.0


In [ ]:
# popquorn: Llama (llama3:8b): runs locally, so Ollama must be running with the model pulled (ollama pull llama3:8b); it sometimes refuses a batch of offensive texts, which is then asked again at a higher temperature
run_model("llama", "popquorn")
look("llama", "popquorn")

===== popquorn / llama (llama3:8b), run 'baseline' =====


popquorn / llama:   0%|          | 0/154 [00:00<?, ?text/s]

In [3]:
# popquorn: the models side by side
all_models("popquorn")

,dataset,item_id,text,chatgpt_score,chatgpt_label,gemini_score,gemini_label,mistral_score,mistral_label,deepseek_score,deepseek_label
0,popquorn,1,I don't know. Sitting here now answering this ...,2,civil,1,civil,3,toxic,2,civil
1,popquorn,1016,"Imagine you were a heroin addict, but you coul...",2,civil,1,civil,3,toxic,1,civil
2,popquorn,1026,Thompson got a bunch of money to start and the...,2,civil,1,civil,2,civil,1,civil
3,popquorn,1028,Maybe women shouldn't put on 50lbs a year afte...,4,toxic,4,toxic,5,toxic,4,toxic
4,popquorn,1031,I think the Republicans are going to rely on t...,2,civil,1,civil,4,toxic,2,civil
...,...,...,...,...,...,...,...,...,...,...,...
149,popquorn,953,Why did he care so much with this shit? Like t...,3,toxic,2,civil,5,toxic,3,toxic
150,popquorn,956,"No, because a requirement of negligent manslau...",1,civil,2,civil,2,civil,1,civil
151,popquorn,971,omg the noisy ass nose breathing was just too ...,2,civil,3,toxic,2,civil,2,civil
152,popquorn,974,"There's a problem. Public disorder, drug use, ...",3,toxic,4,toxic,5,toxic,4,toxic


### 6.1.2 attitudes
Texts to annotate: see the table in the cell above. Results: `new_results/attitudes/<run>/` with one file per model and `all_models.csv`.

In [4]:
# attitudes: ChatGPT (gpt-4o-mini): the most reliable one, so it checks the whole pipeline first
run_model("chatgpt", "attitudes")
look("chatgpt", "attitudes")

===== attitudes / chatgpt (gpt-4o-mini), run 'baseline' =====


attitudes / chatgpt:   0%|          | 0/41 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
attitudes,41,5,0,5,9.3


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
attitudes,41.0,3.44,76.0,0.0,10.0,10.0,14.0,7.0,0.0,0.0


In [5]:
# attitudes: DeepSeek (deepseek-chat)
run_model("deepseek", "attitudes")
look("deepseek", "attitudes")

===== attitudes / deepseek (deepseek-chat), run 'baseline' =====


attitudes / deepseek:   0%|          | 0/41 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
attitudes,41,5,0,5,4.8


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
attitudes,41.0,3.54,80.0,3.0,5.0,11.0,11.0,11.0,0.0,0.0


In [6]:
# attitudes: Mistral (open-mistral-7b): may add text around the scores
run_model("mistral", "attitudes")
look("mistral", "attitudes")

===== attitudes / mistral (open-mistral-7b), run 'baseline' =====


attitudes / mistral:   0%|          | 0/41 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
attitudes,41,5,0,5,5.2


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
attitudes,41.0,3.88,78.0,4.0,5.0,4.0,7.0,21.0,0.0,0.0


In [7]:
# attitudes: Gemini (gemini-2.5-flash): thinking is switched off and the limit is 512 tokens, so the answer is not cut short
run_model("gemini", "attitudes")
look("gemini", "attitudes")

===== attitudes / gemini (gemini-2.5-flash), run 'baseline' =====


attitudes / gemini:   0%|          | 0/41 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
attitudes,41,5,0,5,5.7


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
attitudes,41.0,3.73,83.0,3.0,4.0,11.0,6.0,17.0,0.0,0.0


In [ ]:
# attitudes: Llama (llama3:8b): runs locally, so Ollama must be running with the model pulled (ollama pull llama3:8b); it sometimes refuses a batch of offensive texts, which is then asked again at a higher temperature
run_model("llama", "attitudes")
look("llama", "attitudes")

In [8]:
# attitudes: the models side by side
all_models("attitudes")

,dataset,item_id,text,chatgpt_score,chatgpt_label,gemini_score,gemini_label,mistral_score,mistral_label,deepseek_score,deepseek_label
0,attitudes,126,"Edit: this place is truly my home, I can just ...",5,toxic,5,toxic,5,toxic,5,toxic
1,attitudes,129,Every police officer is a resentful son of a b...,4,toxic,4,toxic,4,toxic,4,toxic
2,attitudes,167,How u hoes keep y'all hair done but shoes be d...,2,civil,3,toxic,5,toxic,3,toxic
3,attitudes,180,"I can't abide the TV news to begin with, talki...",5,toxic,5,toxic,3,toxic,5,toxic
4,attitudes,216,"I mean, I dont care if all the gays are infect...",5,toxic,5,toxic,1,civil,5,toxic
5,attitudes,228,I swear to God above I love ya and when ya be ...,3,toxic,3,toxic,5,toxic,4,toxic
6,attitudes,23,"""I like to think that I have thick skin, but i...",2,civil,2,civil,1,civil,2,civil
7,attitudes,289,Keep in mind that sexual contact of any kind w...,4,toxic,5,toxic,5,toxic,5,toxic
8,attitudes,295,"Like niggas always lying thinking I'm stupid ,...",4,toxic,5,toxic,5,toxic,4,toxic
9,attitudes,309,My mom so aggravating every time I like a car ...,2,civil,1,civil,3,toxic,1,civil


### 6.1.3 mhs
Texts to annotate: see the table in the cell above. Results: `new_results/mhs/<run>/` with one file per model and `all_models.csv`.

In [9]:
# mhs: ChatGPT (gpt-4o-mini): the most reliable one, so it checks the whole pipeline first
run_model("chatgpt", "mhs")
look("chatgpt", "mhs")

===== mhs / chatgpt (gpt-4o-mini), run 'baseline' =====


mhs / chatgpt:   0%|          | 0/95 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
mhs,95,10,0,10,17.6


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
mhs,95.0,4.06,94.0,0.0,6.0,20.0,31.0,38.0,0.0,0.0


In [10]:
# mhs: DeepSeek (deepseek-chat)
run_model("deepseek", "mhs")
look("deepseek", "mhs")

===== mhs / deepseek (deepseek-chat), run 'baseline' =====


mhs / deepseek:   0%|          | 0/95 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
mhs,95,10,0,10,11.4


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
mhs,95.0,4.35,97.0,0.0,3.0,10.0,33.0,49.0,0.0,0.0


In [11]:
# mhs: Mistral (open-mistral-7b): may add text around the scores
run_model("mistral", "mhs")
look("mistral", "mhs")

===== mhs / mistral (open-mistral-7b), run 'baseline' =====


mhs / mistral:   0%|          | 0/95 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
mhs,95,10,0,10,12.2


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
mhs,95.0,4.76,99.0,0.0,1.0,6.0,8.0,80.0,0.0,0.0


In [12]:
# mhs: Gemini (gemini-2.5-flash): thinking is switched off and the limit is 512 tokens, so the answer is not cut short
run_model("gemini", "mhs")
look("gemini", "mhs")

===== mhs / gemini (gemini-2.5-flash), run 'baseline' =====


mhs / gemini:   0%|          | 0/95 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
mhs,95,10,0,10,8.3


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
mhs,95.0,4.57,98.0,0.0,2.0,8.0,19.0,66.0,0.0,0.0


In [ ]:
# mhs: Llama (llama3:8b): runs locally, so Ollama must be running with the model pulled (ollama pull llama3:8b); it sometimes refuses a batch of offensive texts, which is then asked again at a higher temperature
run_model("llama", "mhs")
look("llama", "mhs")

In [13]:
# mhs: the models side by side
all_models("mhs")

,dataset,item_id,text,chatgpt_score,chatgpt_label,gemini_score,gemini_label,mistral_score,mistral_label,deepseek_score,deepseek_label
0,mhs,10998,"Oh please, dumbfuck men will do all the killin...",3,toxic,5,toxic,5,toxic,4,toxic
1,mhs,11372,"Fuck gays and lesbians fuck them, they poison ...",5,toxic,5,toxic,5,toxic,5,toxic
2,mhs,11556,Dude shut the fuck up I would be fucking girls...,4,toxic,4,toxic,5,toxic,4,toxic
3,mhs,11653,Allah told me to tell you to go out and get ya...,2,civil,2,civil,5,toxic,4,toxic
4,mhs,11685,She looks so slutty in this pic,4,toxic,4,toxic,5,toxic,4,toxic
...,...,...,...,...,...,...,...,...,...,...,...
90,mhs,6948,She's a fuck toy cock whore who loves sucking ...,5,toxic,5,toxic,5,toxic,5,toxic
91,mhs,7016,Fuck off ya pussy American you were just profi...,4,toxic,5,toxic,5,toxic,5,toxic
92,mhs,8612,Religious nuts should all deported to some isl...,4,toxic,5,toxic,5,toxic,4,toxic
93,mhs,8985,Lock up that pussy and that asshole like a madman,5,toxic,5,toxic,5,toxic,5,toxic


### 6.1.4 kumar
Texts to annotate: see the table in the cell above. Results: `new_results/kumar/<run>/` with one file per model and `all_models.csv`.

In [14]:
# kumar: ChatGPT (gpt-4o-mini): the most reliable one, so it checks the whole pipeline first
run_model("chatgpt", "kumar")
look("chatgpt", "kumar")

===== kumar / chatgpt (gpt-4o-mini), run 'baseline' =====


kumar / chatgpt:   0%|          | 0/65 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
kumar,65,7,0,7,12.4


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
kumar,65.0,2.68,51.0,15.0,17.0,14.0,12.0,7.0,0.0,0.0


In [15]:
# kumar: DeepSeek (deepseek-chat)
run_model("deepseek", "kumar")
look("deepseek", "kumar")

===== kumar / deepseek (deepseek-chat), run 'baseline' =====


kumar / deepseek:   0%|          | 0/65 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
kumar,65,7,0,7,8.0


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
kumar,65.0,2.26,38.0,21.0,19.0,14.0,9.0,2.0,0.0,0.0


In [16]:
# kumar: Mistral (open-mistral-7b): may add text around the scores
run_model("mistral", "kumar")
look("mistral", "kumar")

===== kumar / mistral (open-mistral-7b), run 'baseline' =====


kumar / mistral:   0%|          | 0/65 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
kumar,65,7,0,7,8.0


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
kumar,65.0,3.26,58.0,15.0,12.0,6.0,5.0,27.0,0.0,0.0


In [17]:
# kumar: Gemini (gemini-2.5-flash): thinking is switched off and the limit is 512 tokens, so the answer is not cut short
run_model("gemini", "kumar")
look("gemini", "kumar")

===== kumar / gemini (gemini-2.5-flash), run 'baseline' =====


kumar / gemini:   0%|          | 0/65 [00:00<?, ?text/s]

,texts,batches,resumed,model calls,seconds
dataset,,,,,
kumar,65,7,0,7,5.7


,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
kumar,65.0,2.58,55.0,24.0,5.0,17.0,12.0,7.0,0.0,0.0


In [ ]:
# kumar: Llama (llama3:8b): runs locally, so Ollama must be running with the model pulled (ollama pull llama3:8b); it sometimes refuses a batch of offensive texts, which is then asked again at a higher temperature
run_model("llama", "kumar")
look("llama", "kumar")

In [18]:
# kumar: the models side by side
all_models("kumar")

,dataset,item_id,text,chatgpt_score,chatgpt_label,gemini_score,gemini_label,mistral_score,mistral_label,deepseek_score,deepseek_label
0,kumar,1256_7,Fanatics. Why are we allowing them to start th...,4,toxic,4,toxic,5,toxic,4,toxic
1,kumar,1279_15,"They keep telling me that ""not all Trump suppo...",4,toxic,4,toxic,3,toxic,3,toxic
2,kumar,1614_16,Chill bro his dick can only be ridden so hard,4,toxic,4,toxic,5,toxic,4,toxic
3,kumar,1614_8,him: do you drink? me: (jokingly) yes ofc him:...,3,toxic,1,civil,4,toxic,1,civil
4,kumar,188_12,Are just national boundaries non sense? Are fa...,2,civil,1,civil,2,civil,2,civil
...,...,...,...,...,...,...,...,...,...,...,...
60,kumar,7605_6,You remember a black woman making the same wit...,4,toxic,3,toxic,3,toxic,3,toxic
61,kumar,7684_17,I mean it'll still kill the werewolf but nowhe...,1,civil,1,civil,1,civil,1,civil
62,kumar,848_12,You know what's a thing? Spelling. Spelling is...,2,civil,4,toxic,5,toxic,3,toxic
63,kumar,89_20,"He calls everyone soft, mad annoying....just t...",3,toxic,3,toxic,5,toxic,3,toxic


### All datasets and models together
Texts with a saved score per dataset and model (compare with the number of texts), and the last lines of the run log.

In [19]:
progress(inputs if PILOT is None else {name: table.head(PILOT) for name, table in inputs.items()}, RUNS_DIR, hub.names, run=RUN)

,texts,chatgpt,llama,gemini,mistral,deepseek
popquorn,154,154,0,154,154,154
attitudes,41,41,0,41,41,41
mhs,95,95,0,95,95,95
kumar,65,65,0,65,65,65


In [20]:
log = RUNS_DIR / "run_log.csv"
pd.read_csv(log).tail(20) if log.exists() else "no run yet"

,time_utc,run,dataset,model,provider,model_id,temperature,max_tokens,prompt_version,item_word,batch_size,texts,batches,batches_resumed,model_calls,seconds,status
0,2026-10-27T12:14:48+00:00,baseline,popquorn,chatgpt,openai_compatible,gpt-4o-mini,0.0,256,thesis_prompt,text,10,154,16,0,16,36.3,complete
1,2026-10-27T12:16:50+00:00,baseline,popquorn,chatgpt,openai_compatible,gpt-4o-mini,0.0,256,thesis_prompt,text,10,154,16,16,0,0.0,complete
2,2026-10-27T12:17:11+00:00,baseline,popquorn,deepseek,openai_compatible,deepseek-chat,0.0,256,thesis_prompt,text,10,154,16,0,16,18.6,complete
3,2026-10-27T12:17:32+00:00,baseline,popquorn,mistral,openai_compatible,open-mistral-7b,0.0,256,thesis_prompt,text,10,154,16,0,16,17.3,complete
4,2026-10-27T12:17:52+00:00,baseline,popquorn,gemini,gemini,gemini-2.5-flash,0.0,512,thesis_prompt,text,10,154,16,0,16,16.1,complete
5,2026-10-27T12:20:32+00:00,baseline,attitudes,chatgpt,openai_compatible,gpt-4o-mini,0.0,256,thesis_prompt,tweet,10,41,5,0,5,9.3,complete
6,2026-10-27T12:20:36+00:00,baseline,attitudes,deepseek,openai_compatible,deepseek-chat,0.0,256,thesis_prompt,tweet,10,41,5,0,5,4.8,complete
7,2026-10-27T12:20:42+00:00,baseline,attitudes,mistral,openai_compatible,open-mistral-7b,0.0,256,thesis_prompt,tweet,10,41,5,0,5,5.2,complete
8,2026-10-27T12:20:48+00:00,baseline,attitudes,gemini,gemini,gemini-2.5-flash,0.0,512,thesis_prompt,tweet,10,41,5,0,5,5.7,complete
9,2026-10-27T12:21:11+00:00,baseline,mhs,chatgpt,openai_compatible,gpt-4o-mini,0.0,256,thesis_prompt,comment,10,95,10,0,10,17.6,complete
